# SMS Spam Classifier: NLP Preprocessing

## Business Context
Raw text messages cannot be fed directly into machine learning models. This notebook converts unstructured text into numerical features that algorithms can understand.

## Notebook Objectives
1. Clean text (lowercase, remove punctuation, remove numbers)
2. Tokenize messages into individual words
3. Remove stopwords (common words with no predictive value)
4. Apply stemming to reduce words to root form
5. Convert text to TF-IDF numerical features
6. Save vectorized data for model training

## Output for Next Notebook
- X_train, X_test: TF-IDF feature matrices
- y_train, y_test: Labels (spam/ham)
- tfidf_vectorizer.pkl: Saved vectorizer for future predictions

#### Import Libraries and Load Data

In [1]:
print('-' * 140)
print('NOTEBOOK 2: NLP PREPROCESSING')
print('-' * 140)

import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
import logging
from datetime import datetime

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

print('Libraries loaded successfully')
logging.info('Notebook 2 started')

--------------------------------------------------------------------------------------------------------------------------------------------
NOTEBOOK 2: NLP PREPROCESSING
--------------------------------------------------------------------------------------------------------------------------------------------


2026-05-09 07:42:33,210 - INFO - Notebook 2 started


Libraries loaded successfully


#### Load Cleaned Data from Notebook 1

In [2]:
print('-' * 140)
print('LOADING CLEANED DATA')
print('-' * 140)

df = pd.read_csv('../data/processed/cleaned_sms.csv')

print(f'Data loaded successfully')
print(f'Total messages: {len(df):,}')
print(f'Columns: {list(df.columns)}')
print(f'\nFirst 5 rows:')
print(df.head())

logging.info(f'Loaded {len(df)} messages for preprocessing')

2026-05-09 07:42:33,303 - INFO - Loaded 5572 messages for preprocessing


--------------------------------------------------------------------------------------------------------------------------------------------
LOADING CLEANED DATA
--------------------------------------------------------------------------------------------------------------------------------------------
Data loaded successfully
Total messages: 5,572
Columns: ['label', 'message']

First 5 rows:
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


#### Step 1: Create Text Cleaning Function

In [3]:
print('-' * 140)
print('STEP 1: TEXT CLEANING FUNCTION')
print('-' * 140)

def clean_text(text):
    # Convert to lowercase
    text = text.lower()
    
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Test the function on a sample message
sample = "WINNER! Call 1800-FREE now to claim your $1000 prize!!!"
cleaned_sample = clean_text(sample)

print(f'Original: {sample}')
print(f'Cleaned: {cleaned_sample}')
print('\nFunction created successfully')
logging.info('Text cleaning function defined')

2026-05-09 07:42:33,350 - INFO - Text cleaning function defined


--------------------------------------------------------------------------------------------------------------------------------------------
STEP 1: TEXT CLEANING FUNCTION
--------------------------------------------------------------------------------------------------------------------------------------------
Original: WINNER! Call 1800-FREE now to claim your $1000 prize!!!
Cleaned: winner call free now to claim your prize

Function created successfully


#### Apply Cleaning Function to All Messages

In [4]:
print('-' * 140)
print('APPLYING TEXT CLEANING TO ALL MESSAGES')
print('-' * 140)

# Apply cleaning function to message column
df['cleaned_message'] = df['message'].apply(clean_text)

print('Before cleaning - sample:')
print(df['message'].iloc[0])
print('\nAfter cleaning - sample:')
print(df['cleaned_message'].iloc[0])

print('\n' + '-' * 140)
print('CLEANING COMPLETE')
print('-' * 140)
print(f'Original messages column retained')
print(f'New column added: cleaned_message')
print(f'Total messages cleaned: {len(df):,}')

logging.info('Text cleaning applied to all messages')

--------------------------------------------------------------------------------------------------------------------------------------------
APPLYING TEXT CLEANING TO ALL MESSAGES
--------------------------------------------------------------------------------------------------------------------------------------------


2026-05-09 07:42:33,518 - INFO - Text cleaning applied to all messages


Before cleaning - sample:
Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...

After cleaning - sample:
go until jurong point crazy available only in bugis n great world la e buffet cine there got amore wat

--------------------------------------------------------------------------------------------------------------------------------------------
CLEANING COMPLETE
--------------------------------------------------------------------------------------------------------------------------------------------
Original messages column retained
New column added: cleaned_message
Total messages cleaned: 5,572


#### Step 2: Tokenization Function

In [5]:
print('-' * 140)
print('STEP 2: TOKENIZATION FUNCTION')
print('-' * 140)

def tokenize_text(text):
    # Split text into individual words
    tokens = text.split()
    return tokens

# Test the function on cleaned sample
test_cleaned = "winner call free now to claim your prize"
tokens = tokenize_text(test_cleaned)

print(f'Cleaned text: {test_cleaned}')
print(f'Tokens: {tokens}')
print(f'Number of tokens: {len(tokens)}')

print('\nTokenization function created successfully')
logging.info('Tokenization function defined')

2026-05-09 07:42:33,545 - INFO - Tokenization function defined


--------------------------------------------------------------------------------------------------------------------------------------------
STEP 2: TOKENIZATION FUNCTION
--------------------------------------------------------------------------------------------------------------------------------------------
Cleaned text: winner call free now to claim your prize
Tokens: ['winner', 'call', 'free', 'now', 'to', 'claim', 'your', 'prize']
Number of tokens: 8

Tokenization function created successfully


#### Step 3: Stopword Removal Function

In [6]:
print('-' * 140)
print('STEP 3: STOPWORD REMOVAL FUNCTION')
print('-' * 140)

# Get English stopwords list
stop_words = set(stopwords.words('english'))

print(f'Total stopwords in English: {len(stop_words)}')
print(f'Sample stopwords: {list(stop_words)[:20]}')

def remove_stopwords(tokens):
    # Remove words that are in stopwords list
    filtered_tokens = [word for word in tokens if word not in stop_words]
    return filtered_tokens

# Test the function
test_tokens = ['winner', 'call', 'free', 'now', 'to', 'claim', 'your', 'prize']
filtered_tokens = remove_stopwords(test_tokens)

print(f'\nOriginal tokens: {test_tokens}')
print(f'After stopword removal: {filtered_tokens}')
print(f'Removed words: {[w for w in test_tokens if w not in filtered_tokens]}')

print('\nStopword removal function created successfully')
logging.info('Stopword removal function defined')

2026-05-09 07:42:33,587 - INFO - Stopword removal function defined


--------------------------------------------------------------------------------------------------------------------------------------------
STEP 3: STOPWORD REMOVAL FUNCTION
--------------------------------------------------------------------------------------------------------------------------------------------
Total stopwords in English: 198
Sample stopwords: ['an', 'from', 'own', 'this', 'doesn', 'you', 'in', "you'll", 're', 'ourselves', 'their', 'how', "you've", 'of', 'are', 'while', 'they', 'him', "we're", "doesn't"]

Original tokens: ['winner', 'call', 'free', 'now', 'to', 'claim', 'your', 'prize']
After stopword removal: ['winner', 'call', 'free', 'claim', 'prize']
Removed words: ['now', 'to', 'your']

Stopword removal function created successfully


#### Step 4: Stemming Function

In [7]:
print('-' * 140)
print('STEP 4: STEMMING FUNCTION')
print('-' * 140)

# Initialize stemmer
stemmer = PorterStemmer()

def stem_words(tokens):
    # Reduce each word to its root form
    stemmed_tokens = [stemmer.stem(word) for word in tokens]
    return stemmed_tokens

# Test the function
test_tokens = ['winner', 'calling', 'free', 'claimed', 'prizes', 'playing']
stemmed_tokens = stem_words(test_tokens)

print(f'Original tokens: {test_tokens}')
print(f'After stemming: {stemmed_tokens}')

print('\nStemming examples:')
print('running ->', stemmer.stem('running'))
print('studies ->', stemmer.stem('studies'))
print('better ->', stemmer.stem('better'))

print('\nStemming function created successfully')
logging.info('Stemming function defined')

2026-05-09 07:42:33,708 - INFO - Stemming function defined


--------------------------------------------------------------------------------------------------------------------------------------------
STEP 4: STEMMING FUNCTION
--------------------------------------------------------------------------------------------------------------------------------------------
Original tokens: ['winner', 'calling', 'free', 'claimed', 'prizes', 'playing']
After stemming: ['winner', 'call', 'free', 'claim', 'prize', 'play']

Stemming examples:
running -> run
studies -> studi
better -> better

Stemming function created successfully


#### Combine All Preprocessing Steps into One Function

In [8]:
print('-' * 140)
print('COMBINING ALL PREPROCESSING STEPS')
print('-' * 140)

def preprocess_message(text):
    # Step 1: Clean text
    text = clean_text(text)
    
    # Step 2: Tokenize
    tokens = text.split()
    
    # Step 3: Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    
    # Step 4: Apply stemming
    tokens = [stemmer.stem(word) for word in tokens]
    
    # Join tokens back into a string
    processed_text = ' '.join(tokens)
    
    return processed_text

# Test the complete pipeline
sample = "WINNER! Call 1800-FREE now to claim your $1000 prize!!!"
processed_sample = preprocess_message(sample)

print('COMPLETE PREPROCESSING PIPELINE TEST:')
print('-' * 70)
print(f'Original: {sample}')
print(f'Processed: {processed_sample}')

print('\nPipeline created successfully')
logging.info('Complete preprocessing pipeline defined')

2026-05-09 07:42:33,738 - INFO - Complete preprocessing pipeline defined


--------------------------------------------------------------------------------------------------------------------------------------------
COMBINING ALL PREPROCESSING STEPS
--------------------------------------------------------------------------------------------------------------------------------------------
COMPLETE PREPROCESSING PIPELINE TEST:
----------------------------------------------------------------------
Original: WINNER! Call 1800-FREE now to claim your $1000 prize!!!
Processed: winner call free claim prize

Pipeline created successfully


#### Apply Preprocessing to All Messages

In [9]:
print('-' * 140)
print('APPLYING PREPROCESSING TO ALL MESSAGES')
print('-' * 140)

# Apply preprocessing to all messages
df['processed_message'] = df['message'].apply(preprocess_message)

# Show comparison
print('BEFORE AND AFTER PREPROCESSING:')
print('-' * 70)
print(f'Original: {df["message"].iloc[0]}')
print(f'Processed: {df["processed_message"].iloc[0]}')

print('\n' + '-' * 70)
print(f'Second example:')
print(f'Original: {df["message"].iloc[1]}')
print(f'Processed: {df["processed_message"].iloc[1]}')

print('\n' + '-' * 140)
print('PREPROCESSING COMPLETE')
print('-' * 140)
print(f'Total messages processed: {len(df):,}')
print(f'New column added: processed_message')

logging.info(f'Preprocessing applied to all {len(df)} messages')

--------------------------------------------------------------------------------------------------------------------------------------------
APPLYING PREPROCESSING TO ALL MESSAGES
--------------------------------------------------------------------------------------------------------------------------------------------


2026-05-09 07:42:34,883 - INFO - Preprocessing applied to all 5572 messages


BEFORE AND AFTER PREPROCESSING:
----------------------------------------------------------------------
Original: Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
Processed: go jurong point crazi avail bugi n great world la e buffet cine got amor wat

----------------------------------------------------------------------
Second example:
Original: Ok lar... Joking wif u oni...
Processed: ok lar joke wif u oni

--------------------------------------------------------------------------------------------------------------------------------------------
PREPROCESSING COMPLETE
--------------------------------------------------------------------------------------------------------------------------------------------
Total messages processed: 5,572
New column added: processed_message


#### Step 5: Convert Text to Numerical Features (TF-IDF)

In [10]:
print('-' * 140)
print('STEP 5: TF-IDF VECTORIZATION')
print('-' * 140)

# Initialize TF-IDF vectorizer
tfidf = TfidfVectorizer(max_features=5000)

# Convert processed messages to TF-IDF features
X = tfidf.fit_transform(df['processed_message']).toarray()

# Get labels (spam = 1, ham = 0)
y = (df['label'] == 'spam').astype(int)

print(f'TF-IDF matrix shape: {X.shape}')
print(f'Number of features: {X.shape[1]}')
print(f'Number of samples: {X.shape[0]}')
print(f'Labels shape: {y.shape}')

print(f'\nSample TF-IDF values for first message:')
print(f'First 10 feature values: {X[0][:10]}')

logging.info(f'TF-IDF vectorization complete. Features: {X.shape[1]}')

--------------------------------------------------------------------------------------------------------------------------------------------
STEP 5: TF-IDF VECTORIZATION
--------------------------------------------------------------------------------------------------------------------------------------------


2026-05-09 07:42:35,158 - INFO - TF-IDF vectorization complete. Features: 5000


TF-IDF matrix shape: (5572, 5000)
Number of features: 5000
Number of samples: 5572
Labels shape: (5572,)

Sample TF-IDF values for first message:
First 10 feature values: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


#### Train-Test Split

In [11]:
print('-' * 140)
print('TRAIN-TEST SPLIT')
print('-' * 140)

# Split data into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set size: {X_train.shape[0]} messages')
print(f'Testing set size: {X_test.shape[0]} messages')
print(f'Training features shape: {X_train.shape}')
print(f'Testing features shape: {X_test.shape}')

print('\nClass distribution in training set:')
print(f'Spam: {y_train.sum()} ({y_train.mean()*100:.2f}%)')
print(f'Ham: {len(y_train) - y_train.sum()} ({(1-y_train.mean())*100:.2f}%)')

print('\nClass distribution in testing set:')
print(f'Spam: {y_test.sum()} ({y_test.mean()*100:.2f}%)')
print(f'Ham: {len(y_test) - y_test.sum()} ({(1-y_test.mean())*100:.2f}%)')

logging.info(f'Train size: {len(X_train)}, Test size: {len(X_test)}')

2026-05-09 07:42:35,341 - INFO - Train size: 4457, Test size: 1115


--------------------------------------------------------------------------------------------------------------------------------------------
TRAIN-TEST SPLIT
--------------------------------------------------------------------------------------------------------------------------------------------
Training set size: 4457 messages
Testing set size: 1115 messages
Training features shape: (4457, 5000)
Testing features shape: (1115, 5000)

Class distribution in training set:
Spam: 598 (13.42%)
Ham: 3859 (86.58%)

Class distribution in testing set:
Spam: 149 (13.36%)
Ham: 966 (86.64%)


#### Save Processed Data and Vectorizer for Next Notebook

In [12]:
print('-' * 140)
print('SAVING PROCESSED DATA AND VECTORIZER')
print('-' * 140)

# Save TF-IDF vectorizer
joblib.dump(tfidf, '../models/tfidf_vectorizer.pkl')

# Save feature matrices and labels as numpy files
np.save('../models/X_train.npy', X_train)
np.save('../models/X_test.npy', X_test)
np.save('../models/y_train.npy', y_train)
np.save('../models/y_test.npy', y_test)

# Save original processed dataframe for reference
df[['message', 'label', 'processed_message']].to_csv('../data/processed/preprocessed_sms.csv', index=False)

print('Files saved successfully:')
print('-' * 70)
print('1. ../models/tfidf_vectorizer.pkl - TF-IDF vectorizer')
print('2. ../models/X_train.npy - Training features')
print('3. ../models/X_test.npy - Testing features')
print('4. ../models/y_train.npy - Training labels')
print('5. ../models/y_test.npy - Testing labels')
print('6. ../data/processed/preprocessed_sms.csv - Preprocessed messages')

logging.info('All data and models saved successfully')

--------------------------------------------------------------------------------------------------------------------------------------------
SAVING PROCESSED DATA AND VECTORIZER
--------------------------------------------------------------------------------------------------------------------------------------------


2026-05-09 07:42:37,535 - INFO - All data and models saved successfully


Files saved successfully:
----------------------------------------------------------------------
1. ../models/tfidf_vectorizer.pkl - TF-IDF vectorizer
2. ../models/X_train.npy - Training features
3. ../models/X_test.npy - Testing features
4. ../models/y_train.npy - Training labels
5. ../models/y_test.npy - Testing labels
6. ../data/processed/preprocessed_sms.csv - Preprocessed messages


#### Notebook 2 Summary

In [13]:
print('-' * 140)
print('NOTEBOOK 2 EXECUTION SUMMARY')
print('-' * 140)

print(f'Execution timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'Total messages processed: {len(df):,}')

print('\nPREPROCESSING STEPS COMPLETED:')
print('-' * 70)
print('1. Text cleaning (lowercase, remove punctuation, remove numbers)')
print('2. Tokenization')
print('3. Stopword removal')
print('4. Stemming')
print('5. TF-IDF vectorization')
print('6. Train-test split (80/20 with stratification)')

print('\nOUTPUT FILES GENERATED:')
print('-' * 70)
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_test shape: {y_test.shape}')
print(f'TF-IDF features: {X.shape[1]}')

print('\n' + '-' * 140)
print('NOTEBOOK 2 COMPLETE - READY FOR MODEL TRAINING')
print('-' * 140)

logging.info('Notebook 2 completed successfully. Ready for Notebook 3 - Model Training')

2026-05-09 07:42:37,554 - INFO - Notebook 2 completed successfully. Ready for Notebook 3 - Model Training


--------------------------------------------------------------------------------------------------------------------------------------------
NOTEBOOK 2 EXECUTION SUMMARY
--------------------------------------------------------------------------------------------------------------------------------------------
Execution timestamp: 2026-05-09 07:42:37
Total messages processed: 5,572

PREPROCESSING STEPS COMPLETED:
----------------------------------------------------------------------
1. Text cleaning (lowercase, remove punctuation, remove numbers)
2. Tokenization
3. Stopword removal
4. Stemming
5. TF-IDF vectorization
6. Train-test split (80/20 with stratification)

OUTPUT FILES GENERATED:
----------------------------------------------------------------------
X_train shape: (4457, 5000)
X_test shape: (1115, 5000)
y_train shape: (4457,)
y_test shape: (1115,)
TF-IDF features: 5000

-------------------------------------------------------------------------------------------------------------